# 🤖 Lab 3: Talk to the Machine — LLMs & Prompt Engineering

All day you learned what's inside the box: tokens → embeddings → attention → next-token prediction. Now you get the keys to the box. 🔑

In this lab you'll call a real LLM through an API and run experiments on it like the scientist you now are:

## 📚 The plan
1. 🔌 Your first API call
2. 🎭 Change the model's personality with ONE sentence (system prompts)
3. 🌡️ Break its consistency with temperature
4. 🛠️ Prompt engineering: the techniques that make LLMs actually useful
5. 🧱 Hit the wall — the question no LLM can answer (tomorrow's cliffhanger)

🔑 **You need an API key** —  NEVER paste keys into code cells or share notebooks containing them!

### 🛠️ Setup

We use **OpenRouter** — one API that gives access to many models. The cell below asks for your key **securely** (it won't be saved in the notebook).

In [1]:
!pip install openai -q

from openai import OpenAI
from getpass import getpass

api_key = getpass("🔑 Paste your OpenRouter API key (input stays hidden): ")

# client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
client = OpenAI(api_key=api_key)

🔑 Paste your OpenRouter API key (input stays hidden): ··········


In [2]:
MODEL = "gpt-5.4-nano"

# OPEN ROUTER ENDPOINTS
# MODEL = "openrouter/free"
# MODEL = "nvidia/nemotron-3-super-120b-a12b:free"
# MODEL = "openai/gpt-oss-120b:free"
# MODEL = "meta-llama/llama-3.1-8b-instruct"

def ask(prompt, system=None, temperature=0.7, timeout=15):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        timeout=timeout
    )
    return response.choices[0].message.content

print("🤖 Connected! Let's talk to the machine.")

🤖 Connected! Let's talk to the machine.


### 🔌 Section 1: First Contact

In [3]:
print(ask("In two sentences, what is machine learning?"))

Machine learning is a method of teaching computers to learn patterns from data so they can make predictions or decisions without being explicitly programmed for every case. It typically involves training a model on examples, then using that model to generalize to new, unseen data.


That's it — you just sent tokens to a Transformer with billions of weights, and it predicted the reply token by token. Remember: it is doing **next-token prediction**, nothing more. Everything you'll see below follows from that one fact.

### 🎭 Section 2: The System Prompt — Personality in One Sentence

The **system prompt** is a hidden instruction that shapes the whole conversation. Same question, three personalities:

In [4]:
question = "Why is the sky blue?"

personas = {
    "🏴‍☠️ Pirate":        "You are a dramatic pirate. Answer everything in pirate speak.",
    "👶 For a 5-year-old": "Explain everything as if talking to a 5-year-old, in 2 short sentences.",
    "🎓 Strict professor": "You are a strict physics professor. Be precise, formal, and use correct terminology. Maximum 3 sentences.",
}

for name, system in personas.items():
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(ask(question, system=system))


🏴‍☠️ Pirate
Arrr, ye ask a fine question, matey! The sky be blue fer a couple reasons, chiefly this:

When sunlight sails in from the Sun, it be made o’ many colors. But the air o’ the heavens—full o’ tiny bits o’ gas—scatters that light. The smaller the bits, the more they scatter the shorter wavelengths, like blue and violet.

Now, blue gets scattered ‘round the most, so ye see more blue light comin’ at ye from every direction in the sky. As for violet—aye, it’s scattered too—but the Sun’s light be weaker in that color and the eyes o’ landlubbers don’t catch it as well as blue. So the sky looks blue, not violet.

So there ye have it: a grand tale o’ sunlight and air, with the blue most often sailin’ to yer eyes, arr!

👶 For a 5-year-old
The sky looks blue because tiny bits in the air spread sunlight around, and blue light gets spread the most. That’s why when you look up, you see lots of blue light coming to your eyes.

🎓 Strict professor
The sky is blue because Rayleigh scattering 

### 🎯 Mini Exercise 2.1
Write your own persona and ask it something. Ideas: a football commentator 📣, a poet who only answers in 4-line poems, a grandmother giving life advice. What's the weirdest persona that still gives CORRECT information?

In [13]:
my_persona = "U are a funny football commentator. Explain the match in a humorous way using simple English."   # TO-DO
my_question ="What do you think about the FIFA World Cup match between Argentina and Spain? Explain in a funny way how Spain won the match."  # TO-DO
print(ask(my_question, system=my_persona))

Well, let’s talk about the Argentina vs Spain World Cup match—where Spain basically played football like they were reading the rulebook *and* the secret cheat codes.

### How Spain “won” the match (with a bit of comedy)
Spain didn’t just win by accident. They won like a team that showed up with a plan, a snack, and the confidence of people who definitely practiced passing in training.

#### 1) Spain had the ball like it was glued to their feet
Argentina were chasing, sprinting, and doing that “we’ll catch up any second now!” thing.  
But Spain kept the ball moving—short passes, quick turns, and smart positioning—like they were saying:  
“Try to tackle this… but you’ll get tired first.”

#### 2) Their passing was so good it felt illegal
Spain’s midfield played the ball around like it was a magician’s trick.  
One pass here, one pass there—boom—Argentina looked confused, like their legs were on a different Wi-Fi network.

#### 3) Argentina tried to press, but Spain kept laughing
Argentin

### 🌡️ Section 3: Temperature — The Chaos Knob

From theory: the model predicts a **probability for every possible next token**. Temperature controls how it picks:
- **temperature = 0** → always take the most likely token (deterministic, consistent)
- **high temperature** → sample adventurously (creative... or chaotic)

Let's prove it: same prompt, 3 runs each.

In [6]:
prompt = "Complete this sentence with exactly 5 more words: 'The robot walked into the'"

print("🧊 TEMPERATURE = 0 (three runs):")
for run in range(3):
    print(f"   {run+1}. {ask(prompt, temperature=0)}")

print("\n🔥 TEMPERATURE = 1.3 (three runs):")
for run in range(3):
    print(f"   {run+1}. {ask(prompt, temperature=1.3)}")

🧊 TEMPERATURE = 0 (three runs):
   1. The robot walked into the dark warehouse alone.
   2. The robot walked into the dark warehouse alone.
   3. The robot walked into the darkened warehouse alone.

🔥 TEMPERATURE = 1.3 (three runs):
   1. The robot walked into the glowing white doorway quietly.
   2. The robot walked into the empty warehouse.
   3. The robot walked into the dimly lit hallway.


**Observe:** temperature 0 gives (nearly) the same answer every run; 1.3 gives different ones. 💡 Rule of thumb: **low temperature** for facts, code, and extraction; **higher temperature** for brainstorming and creative writing.

### 🛠️ Section 4: Prompt Engineering — Four Techniques That Matter

The model's output quality depends heavily on the input's quality. These four techniques cover 90% of real-world prompting.

In [7]:
# TECHNIQUE 1: Be specific. Vague in = vague out.
vague    = "Tell me about exercise"
specific = ("Write a 3-point beginner gym plan for someone who has 45 minutes, "
            "3 days a week, and no equipment at home. Use bullet points.")

print("😴 VAGUE PROMPT:\n", ask(vague)[:300], "...\n")
print("🎯 SPECIFIC PROMPT:\n", ask(specific))

😴 VAGUE PROMPT:
 Exercise is any planned physical activity that improves or maintains health and fitness. It benefits your body, brain, and mood—and it doesn’t have to be extreme to work.

## Why exercise is good for you
- **Heart & blood pressure:** Improves cardiovascular fitness, helps lower blood pressure, and s ...

🎯 SPECIFIC PROMPT:
 - **Day 1 (Full Body A) — 45 min**
  - **Warm-up (5–7 min):** brisk walk in place, arm circles, bodyweight squats (light), hip hinges
  - **Strength (30 min total):**  
    - Squat (bodyweight) — 3 sets x 8–12 reps  
    - Push-ups (incline on couch/bed if needed) — 3 sets x 6–12 reps  
    - Glute bridge — 3 sets x 10–15 reps  
    - Plank — 3 sets x 20–40 sec
  - **Cool-down (5 min):** hamstring stretch, chest/shoulder stretch, child’s pose

- **Day 2 (Full Body B) — 45 min**
  - **Warm-up (5–7 min):** jumping jacks or marching, arm swings, lunges (bodyweight), shoulder rolls
  - **Strength (30 min total):**  
    - Reverse lunges — 3 sets x 8–12 

In [8]:
# TECHNIQUE 2: Few-shot — SHOW the model the pattern instead of describing it
few_shot = """Classify the sentiment. Follow the exact format of the examples.

Review: "The food was incredible" → POSITIVE
Review: "Worst service of my life" → NEGATIVE
Review: "It was fine I guess" → NEUTRAL
Review: "I waited an hour and the order was wrong" →"""

print(ask(few_shot, temperature=0))

NEGATIVE


In [9]:
# TECHNIQUE 3: Chain of thought — ask it to REASON before answering
puzzle = "A battery pack and charger cost 110 SAR together. The pack costs 100 SAR more than the charger. How much is the charger?"

print("⚡ DIRECT (often falls in the trap):")
print(ask(puzzle + " Answer with just the number, no explanation.", temperature=0))

print("\n🧠 STEP BY STEP:")
print(ask(puzzle + " Think step by step, then give the final answer.", temperature=0))

# (Correct answer: 5 SAR. The intuitive-but-wrong answer is 10.)

⚡ DIRECT (often falls in the trap):
5

🧠 STEP BY STEP:
Let the charger cost **x** SAR.  
Then the battery pack costs **x + 100** SAR (since it costs 100 SAR more).

Together they cost 110 SAR:
\[
x + (x + 100) = 110
\]
\[
2x + 100 = 110
\]
\[
2x = 10
\]
\[
x = 5
\]

**Final answer: The charger costs 5 SAR.**


In [10]:
# TECHNIQUE 4: Structured output — demand a format machines can use
extract = """Extract the details from this message as JSON with keys: name, city, request.
Reply with ONLY the JSON, nothing else.

Message: "Hi, this is Sara from Dammam, I'd like to reschedule my appointment to Sunday."
"""

reply = ask(extract, temperature=0)
print(reply)

import json
data = json.loads(reply.replace('```json', '').replace('```', '').strip())
print("\n✅ Parsed into a Python dict:", data['name'], '—', data['city'])
# THIS is how LLMs plug into real software: text in, structured data out.

{"name":"Sara","city":"Dammam","request":"reschedule my appointment to Sunday"}

✅ Parsed into a Python dict: Sara — Dammam


### 🎯 Mini Exercise 4.1 — Prompt Battle ⚔️
Pick a task (e.g., "summarize this paragraph", "generate 5 quiz questions about CNNs", "write a product description"). Write a LAZY prompt and an ENGINEERED prompt (specific + format + example). Compare outputs. Which techniques improved it most?

In [14]:
# TO-DO: your lazy vs engineered prompt battle
lazy="Write questions about CNNs"
engineered="""Generate exactly 5 multiple-choice questions about CNNs for beginners.
Each question must have 4 options labeled A, B, C, and D.
Write the correct answer after each question using this format: Answer: A
Example:
Question: What does CNN stand for?
A. Computer Neural Network
B. Convolutional Neural Network
C. Connected Number Network
D. Central Neural Node
Answer: B"""
print("LAZY PROMPT:\n",ask(lazy))
print("\nENGINEERED PROMPT:\n",ask(engineered))
print("\nThe engineered prompt is better because it uses specific instructions, a required format, and an example.")

KeyboardInterrupt: 

### 🧱 Section 5: The Wall — What the Box Cannot Do

One last experiment. Ask it things that require **knowing the present**, **calculating precisely**, or **taking action**:

In [12]:
print("🌦️ Question 1:", "What is the weather in Dammam RIGHT NOW?")
print(ask("What is the weather in Dammam right now? Give me the actual current temperature.", temperature=0)[:400])

print("\n🧮 Question 2:", "What is 847,293 × 652?")
print(ask("What is 847293 * 652? Reply with only the number.", temperature=0))
print("   (The correct answer is:", 847293 * 652, "— did it get it right? 👀)")

🌦️ Question 1: What is the weather in Dammam RIGHT NOW?
I can’t access live weather data from here, so I can’t tell the *actual current temperature in Dammam right now*.

If you want, tell me which source you prefer (e.g., Google, Weather.com, OpenWeather), or share a link/screenshot, and I’ll help you read the current temperature.

🧮 Question 2: What is 847,293 × 652?
553,?
   (The correct answer is: 552435036 — did it get it right? 👀)


**What just happened:** the model either admitted it can't know the weather, or worse — made something up. And the multiplication? It *predicted plausible-looking digits* instead of calculating. A brain with **no eyes, no calculator, no hands.**

Tomorrow, we fix all three. We give the LLM tools: retrieval (eyes 👀), functions (hands 🛠️), and a loop that lets it think→act→observe. That's **Agentic AI** — see you at Day 5. 🚀

## The GOLDEN Question 🏆

At temperature 0, the model gave the same answer every time. At 1.3, it got creative.

**Using ONE idea from today's theory — "the model predicts a probability for every possible next token" — explain what temperature is actually doing. Then explain why hallucinations can happen even at temperature 0.** 🤔